# UCS420 — Cognitive Computing Lab Assignment 4
**Author:** meet-the-1337 | **Roll Number:** 1024171337 | **Branch:** COPC

Design, evaluation, and ranking algorithms for a Personalized Cognitive FAQ Knowledge Base.


## Q1: Build Your Personalized Knowledge Base
Build a pandas DataFrame with 6 FAQ entries (4 fixed + 2 personalized based on last digits of roll number). Columns: `id`, `question`, `answer`, `category`, `keywords`.


In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

roll_number = "1024171337"
digits = [int(d) for d in roll_number]
print("Roll number:", roll_number)
print("Digits:", digits)

last_two = roll_number[-2:]
print("Personalized digit suffixes:", last_two)

faq_data = [
    {
        "id": 1,
        "question": "what are the library opening hours",
        "answer": "The central library is open Monday through Saturday from 8:00 AM to 10:00 PM.",
        "category": "facilities",
        "keywords": "library hours timings open close schedule"
    },
    {
        "id": 2,
        "question": "how do i submit laboratory assignments on github",
        "answer": "Push your Jupyter notebooks to your personal GitHub repository before the weekly deadline.",
        "category": "academics",
        "keywords": "assignment lab submit submission github repository code"
    },
    {
        "id": 3,
        "question": "how do i connect to campus wifi",
        "answer": "Select the campus secure SSID and authenticate with your student network credentials.",
        "category": "technical",
        "keywords": "wifi internet connect network wireless access credentials"
    },
    {
        "id": 4,
        "question": "where can i find the academic calendar and exam datesheet",
        "answer": "The academic calendar and end-semester datesheet are posted on the student web portal.",
        "category": "academics",
        "keywords": "exam datesheet calendar schedule semester dates"
    },
    {
        "id": 5,
        "question": f"how do i update account profile details for student {last_two[0]}",
        "answer": "Navigate to profile settings on the student management portal and verify contact information.",
        "category": "account",
        "keywords": "account profile update contact email phone settings"
    },
    {
        "id": 6,
        "question": f"how do i contact it technical support helpdesk {last_two[1]}",
        "answer": "Send an email to helpdesk@university.edu or visit the IT support desk during lab hours.",
        "category": "technical",
        "keywords": "support helpdesk contact it technical ticket issue"
    }
]

df_faq = pd.DataFrame(faq_data)
print("Knowledge Base DataFrame:")
print(df_faq[["id", "category", "question", "keywords"]])


Roll number: 1024171337
Digits: [1, 0, 2, 4, 1, 7, 1, 3, 3, 7]
Personalized digit suffixes: 37
Knowledge Base DataFrame:
   id    category                                           question                                           keywords
0   1  facilities                 what are the library opening hours          library hours timings open close schedule
1   2   academics   how do i submit laboratory assignments on github  assignment lab submit submission github reposi...
2   3   technical                    how do i connect to campus wifi  wifi internet connect network wireless access ...
3   4   academics  where can i find the academic calendar and exa...    exam datesheet calendar schedule semester dates
4   5     account  how do i update account profile details for st...  account profile update contact email phone set...
5   6   technical   how do i contact it technical support helpdesk 7  support helpdesk contact it technical ticket i...


## Q2: Generate and Score a Hypothesis
A scoring function where keyword matches count 2x higher than general question-word matches.


In [2]:
def score_entries(query, df):
    query_words = set(query.lower().split())
    results = []
    for _, row in df.iterrows():
        keyword_words = set(row["keywords"].lower().split())
        question_words = set(row["question"].lower().split())
        
        # Keyword matches weighted 2x, question matches weighted 1x
        kw_matches = len(query_words.intersection(keyword_words))
        q_matches = len(query_words.intersection(question_words))
        total_score = (kw_matches * 2) + q_matches
        
        results.append({
            "id": row["id"],
            "score": total_score,
            "category": row["category"],
            "question": row["question"],
            "answer": row["answer"]
        })
    results.sort(key=lambda x: x["score"], reverse=True)
    return results

print("Query: 'how to submit lab assignment on github'")
matches = score_entries("how to submit lab assignment on github", df_faq)
for m in matches[:3]:
    print(f"  Score: {m['score']} | Category: {m['category']} | Q: {m['question']}")
    print(f"    Answer: {m['answer']}")


Query: 'how to submit lab assignment on github'
  Score: 12 | Category: academics | Q: how do i submit laboratory assignments on github
    Answer: Push your Jupyter notebooks to your personal GitHub repository before the weekly deadline.
  Score: 2 | Category: technical | Q: how do i connect to campus wifi
    Answer: Select the campus secure SSID and authenticate with your student network credentials.
  Score: 1 | Category: account | Q: how do i update account profile details for student 3
    Answer: Navigate to profile settings on the student management portal and verify contact information.


## Q3: Questions by Category
`same_category(category_name, df)` returns all questions belonging to a given category.


In [3]:
def same_category(category_name, df):
    return df[df["category"] == category_name]["question"].tolist()

print("Questions in category 'academics':")
for q in same_category("academics", df_faq):
    print(f"  - {q}")

print("\nQuestions in category 'technical':")
for q in same_category("technical", df_faq):
    print(f"  - {q}")


Questions in category 'academics':
  - how do i submit laboratory assignments on github
  - where can i find the academic calendar and exam datesheet
Questions in category 'technical':
  - how do i connect to campus wifi
  - how do i contact it technical support helpdesk 7


## Q4: Add a Keyword and Save to CSV
Add a new keyword to a target question and save the updated DataFrame to `1024171337_faq_data.csv`.


In [4]:
new_keyword = "portal_hotspot"
print("Adding keyword:", new_keyword)

target_id = 3
current_kw = df_faq.loc[df_faq["id"] == target_id, "keywords"].values[0]
df_faq.loc[df_faq["id"] == target_id, "keywords"] = current_kw + " " + new_keyword

output_csv = "1024171337_faq_data.csv"
df_faq.to_csv(output_csv, index=False)
print(f"Successfully saved updated knowledge base to '{output_csv}'.")

# Verify written CSV
reloaded = pd.read_csv(output_csv)
print("\nReloaded CSV verification (Row 3):")
print(reloaded.loc[reloaded["id"] == target_id, ["id", "question", "keywords"]])


Adding keyword: portal_hotspot
Successfully saved updated knowledge base to '1024171337_faq_data.csv'.
Reloaded CSV verification (Row 3):
   id                         question                                           keywords
2   3  how do i connect to campus wifi  wifi internet connect network wireless access ...


## Q5: FAQ Entries Per Category
Using `groupby` to count entries per category.


In [5]:
category_counts = df_faq.groupby("category").size()
print("FAQ Entries per Category:")
print(category_counts)


FAQ Entries per Category:
category
academics     2
account       1
facilities    1
technical     2
dtype: int64


## Q6: Handling Ties in the Scoring Function
When two or more entries tie for the highest score, all of them are returned.


In [6]:
def score_entries_with_ties(query, df):
    scored = score_entries(query, df)
    if not scored:
        return []
    max_score = scored[0]["score"]
    if max_score == 0:
        return []
    ties = [item for item in scored if item["score"] == max_score]
    return ties

print("Query yielding a tie: 'technical support schedule'")
tied_results = score_entries_with_ties("technical support schedule", df_faq)
print(f"Found {len(tied_results)} top matching entries sharing score {tied_results[0]['score'] if tied_results else 0}:")
for item in tied_results:
    print(f"  [ID {item['id']}] Q: {item['question']}")


Query yielding a tie: 'technical support schedule'
Found 1 top matching entries sharing score 6:
  [ID 6] Q: how do i contact it technical support helpdesk 7
